# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. 

Adapted from your existing exercise. Previous local outputs were cleared so new Colab runs provide their own evidence. 

Source: Original local package: 02_mit_6s191/03_wearable_lstm_gru_attention.ipynb

In [ ]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '02_mit_6s191', 'torch', '03_wearable_lstm_gru_attention.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

# LO2–LO3: wearable-sequence adaptation of MIT Lab 1
An original LSTM–GRU teaching baseline with temporal attention, modality dropout, class-weighted loss, subject-disjoint validation, checkpoints and missing-modality evaluation. Tensor shape: `(batch, time, three modalities)` → recurrent states → attention pool → three class logits.
**All data and results are synthetic. This is not the Nandini benchmark or AM-LSTM-GRU replication.** It demonstrates components that can inform that work after the exact paper and dataset protocols are implemented.
Source concepts: https://introtodeeplearning.com/


In [ ]:
import copy,json,os
from pathlib import Path
import numpy as np,pandas as pd,torch
from torch import nn
from wearable_utils import *
torch.manual_seed(42);torch.set_num_threads(1)
x,y,subjects=make_wearable_windows()
train=np.flatnonzero(subjects<5);valid=np.flatnonzero(subjects==5);test=np.flatnonzero(subjects>5)
assert set(subjects[train]).isdisjoint(set(subjects[valid])|set(subjects[test]))
scaler=fit_signal_scaler(x[train]);xx=torch.tensor(transform_signals(scaler,x));yy=torch.tensor(y,dtype=torch.long)
results=Path('results');results.mkdir(exist_ok=True)
class WearableRNN(nn.Module):
    def __init__(self,attention=True,drop_probability=.15):
        super().__init__();self.use_attention=attention;self.drop_probability=drop_probability
        self.lstm=nn.LSTM(3,24,batch_first=True)
        self.gru=nn.GRU(24,16,batch_first=True)
        self.attention=nn.Linear(16,1);self.classifier=nn.Linear(16,3)
    def forward(self,x,return_attention=False):
        if self.training and self.drop_probability:
            keep=torch.rand(x.shape[0],1,x.shape[-1],device=x.device)>self.drop_probability
            # Never remove all modalities from an example.
            empty=~keep.any(dim=2,keepdim=True)
            keep=keep | empty
            x=x*keep
        h,_=self.lstm(x);h,_=self.gru(h)
        weights=torch.softmax(self.attention(h),dim=1)
        pooled=(h*weights).sum(dim=1) if self.use_attention else h.mean(dim=1)
        logits=self.classifier(pooled)
        return (logits,weights) if return_attention else logits
counts=np.bincount(y[train],minlength=3)
loss_fn=nn.CrossEntropyLoss(weight=torch.tensor(len(train)/(3*counts),dtype=torch.float32))

In [ ]:
def train_model(attention,drop_probability):
    torch.manual_seed(42);model=WearableRNN(attention,drop_probability)
    optimizer=torch.optim.Adam(model.parameters(),lr=.008);best=-1;state=None;history=[]
    for epoch in range(12):
        model.train();losses=[]
        for batch in torch.as_tensor(train)[torch.randperm(len(train))].split(32):
            optimizer.zero_grad(set_to_none=True);loss=loss_fn(model(xx[batch]),yy[batch]);loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.);optimizer.step();losses.append(float(loss.detach()))
        model.eval()
        with torch.no_grad():pred=model(xx[valid]).argmax(-1).numpy()
        score=classification_report_values(y[valid],pred)['macro_f1']
        history.append({'epoch':epoch,'training_loss':np.mean(losses),'validation_macro_f1':score})
        if score>best:best=score;state=copy.deepcopy(model.state_dict())
    model.load_state_dict(state);model.eval();return model,history,best
models={};rows=[]
for name,attention,drop in [('mean_pool',False,0.),('attention',True,0.),('attention_modality_dropout',True,.15)]:
    model,history,best=train_model(attention,drop);models[name]=model
    pd.DataFrame(history).to_csv(results/f'wearable_{name}_training.csv',index=False)
    rows.append({'model':name,'validation_macro_f1':best})
selection=pd.DataFrame(rows);selected_name=selection.loc[selection.validation_macro_f1.idxmax(),'model']
model=models[selected_name]
torch.save(model.state_dict(),results/'wearable_selected.pt')
(results/'wearable_config.json').write_text(json.dumps({'seed':42,'selected_model':selected_name,
 'train_subjects':[0,1,2,3,4],'valid_subjects':[5],'test_subjects':[6,7],
 'dataset':'synthetic','epochs':12,'learning_rate':.008},indent=2))
print(selection.to_string(index=False))

In [ ]:
report=[]
for condition,missing in [('all modalities',None),('missing modality 0',0),('missing modality 1',1),('missing modality 2',2)]:
    inputs=xx[test].clone()
    if missing is not None:inputs[:,:,missing]=0 # training-standardised mean
    with torch.no_grad():logits,weights=model(inputs,return_attention=True)
    assert torch.allclose(weights.sum(dim=1),torch.ones(len(test),1),atol=1e-5)
    report.append({'condition':condition,**classification_report_values(y[test],logits.argmax(-1).numpy())})
report=pd.DataFrame(report);report.to_csv(results/'synthetic_missing_modality_metrics.csv',index=False)
print(report.to_string(index=False))
print('Attention is a learned weighting mechanism, not a causal explanation.')

Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [ ]:
finish_run(COLAB_CONTEXT)